In [1]:
import xarray as xr
old = xr.open_dataset("/Odyssey/public/_transit_zone_/mask-A.nc")

print(float(old.longitude.min()), float(old.longitude.max()))


/Odyssey/private/n23nguye/envs/nga/miniconda/envs/4dvarnet-gm/lib/python3.11/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)
/Odyssey/private/n23nguye/envs/nga/miniconda/envs/4dvarnet-gm/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1.1e-05 359.99994999999996


In [2]:
import re
import xarray as xr
import pandas as pd
from pathlib import Path

In [3]:
# MY SARAL/AltiKa GEODETIC (alg) -- spans 2015-2025, so it HAS 2024 (al is only 2013-2015)
L3_ROOT = Path('/Odyssey/public/data_challenge_2024/REPROCESSING_L3_KARIN_V3.0/SEALEVEL_GLO_PHY_L3_MY_008_062')
alg_dir = next(L3_ROOT.glob('cmems_obs-sl_glo_phy-ssh_my_alg-l3-duacs_PT1S*'))
print('AltiKa geodetic (MY alg) dir:', alg_dir)
print('subdirs:', sorted(p.name for p in alg_dir.iterdir() if p.is_dir()))   # should include 2024

all_files = sorted(alg_dir.glob('**/*.nc'))
print(f'{len(all_files)} total .nc;  first {all_files[0].name}  last {all_files[-1].name}')
assert all_files

def data_year(name):                       # first 8-digit token = data date (2nd = production date)
    m = re.search(r'(\d{8})', name)
    return m.group(1)[:4] if m else None

files = sorted(p for p in all_files if data_year(p.name) == '2024') or all_files
print(f'{len(files)} candidate files')

ref = xr.open_mfdataset(
    files,
    preprocess=lambda d: d.where(
        (d.time.load() >= pd.to_datetime('2024-01-01')) & (d.time <= pd.to_datetime('2024-12-31')),
        drop=True),
    concat_dim='time', combine='nested',
).sortby('time')

n = int(ref.sizes['time'])
print(f'2024 AltiKa (alg) points: {n:,}')
assert n > 0, 'No 2024 in MY alg -> check subdirs above'
print('  time:', str(ref.time.min().values)[:10], '->', str(ref.time.max().values)[:10])
print('  lon : {:.3f} .. {:.3f}'.format(float(ref.longitude.min()), float(ref.longitude.max())))
print('  vars:', list(ref.data_vars))
for need in ('sla_unfiltered', 'lwe'):
    assert need in ref.data_vars, f'missing {need!r}'


AltiKa geodetic (MY alg) dir: /Odyssey/public/data_challenge_2024/REPROCESSING_L3_KARIN_V3.0/SEALEVEL_GLO_PHY_L3_MY_008_062/cmems_obs-sl_glo_phy-ssh_my_alg-l3-duacs_PT1S_202411
subdirs: ['2023', '2024', '2025']
952 total .nc;  first dt_global_alg_phy_l3_1hz_20230101_20240205.nc  last dt_global_alg_phy_l3_1hz_20250802_20260106.nc
366 candidate files
2024 AltiKa (alg) points: 14,980,299
  time: 2024-01-01 -> 2024-12-30
  lon : 0.000 .. 360.000
  vars: ['cycle', 'track', 'sla_unfiltered', 'sla_filtered', 'dac', 'ocean_tide', 'internal_tide', 'lwe', 'mdt', 'tpa_correction']


In [4]:
ref.to_netcdf('/Odyssey/private/n23nguye/2023a_SSH_mapping_OSE/Notebooks/maskA_altika_2024.nc')
print('wrote maskA_altika_2024.nc')

/Odyssey/private/n23nguye/envs/nga/miniconda/envs/4dvarnet-gm/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3701: SerializationWarning: saving variable latitude with floating point data as an integer dtype without any _FillValue to use for NaNs
  exec(code_obj, self.user_global_ns, self.user_ns)
/Odyssey/private/n23nguye/envs/nga/miniconda/envs/4dvarnet-gm/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3701: SerializationWarning: saving variable longitude with floating point data as an integer dtype without any _FillValue to use for NaNs
  exec(code_obj, self.user_global_ns, self.user_ns)


wrote maskA_altika_2024.nc
